# Stage 1 — QLoRA Supervised Fine-Tuning on PokerBench

Fine-tunes **Qwen3-8B** on 563k solver-verified poker scenarios using QLoRA.

**Runtime:** GPU — T4 (free Colab) or better  
**Expected time:** ~2 hrs on T4 for 1 full epoch  
**Output:** LoRA adapter saved to Google Drive at `MyDrive/pokerapp/sft-adapter/`

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Make your GitHub repo **public**, or add a `GITHUB_TOKEN` secret (Colab sidebar → Secrets)

## 1. Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found — change runtime to T4"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
!pip install -q --upgrade unsloth
!pip install -q datasets trl peft accelerate

## 3. Mount Google Drive

The adapter will be saved here so it persists after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_DIR = "/content/drive/MyDrive/pokerapp/sft-adapter"
CHECKPOINT_DIR = "/content/drive/MyDrive/pokerapp/sft-checkpoints"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Adapter will be saved to    : {ADAPTER_DIR}")
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## 4. Clone repo

Needed to import `src.preprocessor`. If your repo is private, add a `GITHUB_TOKEN` Colab secret.

In [ ]:
import os, sys

# If repo is private, use token from Colab secrets
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/dominicvdb/pokerapp.git"
except Exception:
    repo_url = "https://github.com/dominicvdb/pokerapp.git"

!git clone {repo_url} /content/pokerapp --quiet
sys.path.insert(0, "/content/pokerapp")
print("Repo cloned — src/ is importable")

## 5. Load dataset

In [ ]:
from src.data_loader import load_pokerbench

dataset = load_pokerbench(cache_dir="/content/pokerapp/data")
train_ds = dataset["train"]

print(f"Train rows : {len(train_ds):,}")
print(f"Test rows  : {len(dataset['test']):,}")
print(f"Columns    : {train_ds.column_names}")
print()
print("Example output:", train_ds[0]["output"])

## 6. Load Qwen3-8B in 4-bit with Unsloth

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-8B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # auto-detect bf16/fp16
)

print("Model loaded")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 7. Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
)

model.print_trainable_parameters()

## 8. Preprocess dataset into SFT chat format

In [ ]:
from src.preprocessor import preprocess_sft

sft_dataset = preprocess_sft(train_ds, tokenizer)

print(f"Processed {len(sft_dataset):,} rows")
print()
print("Sample (truncated):")
print(sft_dataset[0]["text"][:400], "...")

## 9. Train with SFTTrainer

- **Full epoch** (~18 hrs on A100 High RAM): leave `max_steps=-1`
- **Quick smoke test** (~5 min): set `max_steps=100`

Checkpoints are saved to Google Drive every 1000 steps. If the session dies, re-run all cells from the top — the trainer will automatically resume from the last checkpoint.

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

# Set max_steps=100 for a quick sanity check, -1 for full training
MAX_STEPS = -1

# T4 (Turing) supports fp16 only; A100/A10G (Ampere+) support bf16
use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"Precision: {'bf16' if use_bf16 else 'fp16'}  (GPU: {torch.cuda.get_device_name(0)})")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    args=SFTConfig(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        max_seq_length=MAX_SEQ_LENGTH,
        bf16=use_bf16,
        fp16=use_fp16,
        warmup_steps=500,
        lr_scheduler_type="cosine",
        logging_steps=50,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=2,
        dataset_text_field="text",
        report_to="none",
    ),
)

print(f"Steps per epoch : {len(trainer.get_train_dataloader()):,}")
print(f"Effective batch : 32")

In [ ]:
import os

# Auto-resume from latest checkpoint if one exists on Drive
checkpoints = [
    d for d in os.listdir(CHECKPOINT_DIR)
    if d.startswith("checkpoint-")
] if os.path.exists(CHECKPOINT_DIR) else []

resume_from = CHECKPOINT_DIR if checkpoints else None
if resume_from:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    print(f"Resuming from checkpoint: {latest}")
else:
    print("Starting from scratch")

trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

print(f"\nTraining complete")
print(f"Runtime : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Loss    : {trainer_stats.metrics['train_loss']:.4f}")

## 10. Save adapter to Google Drive

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to {ADAPTER_DIR}")
!ls -lh {ADAPTER_DIR}

## 11. Quick eval on test samples

Spot-checks 20 examples from the test set to sanity-check the adapter before full evaluation.

In [ ]:
import re
from src.preprocessor import format_grpo, apply_chat_template
from src.reward import poker_reward

FastLanguageModel.for_inference(model)

_think_re = re.compile(r"<think>.*?</think>", re.DOTALL)

test_ds = dataset["test"].select(range(20))
correct = 0

for row in test_ds:
    prompt = apply_chat_template(
        format_grpo(row), tokenizer, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Strip Qwen3 thinking tokens before scoring
    generated_clean = _think_re.sub("", generated).strip()

    reward = poker_reward(generated_clean, row["output"])
    if reward == 1.0:
        correct += 1
    print(f"Expected: {row['output']:<12}  Predicted: {generated_clean:<12}  Reward: {reward}")

print(f"\nSpot-check accuracy: {correct}/20 ({correct * 5}%)")